# Yearly Calculations for variance decomposition prep
Run this cell to calculate the yearly AR-Day extreme precipitation for each ARDT/BC/model configuration.

In [1]:
# Import appropriate modules
import xarray as xa
import numpy as np
from glob import glob
import pandas as pd
import geopandas as gpd
import cftime
import regionmask
import sys
import dask
import os
import dask
from dask.distributed import LocalCluster, Client, progress
import gc
import warnings
warnings.filterwarnings('ignore')

ERROR 1: PROJ: proj_create_from_database: Open of /glade/u/home/tcorrie/.conda/envs/rmask/share/proj failed


In [4]:
# Load original WUS-D3 dataset and add CESM2-LE 
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col=0).drop(index=6).reset_index(drop=True)
for mm in np.arange(1011, 1201, 20):
    wusd3.loc[len(wusd3)]=['cesm2-le', 'CESM2-LE', mm, '365_day']

wusd3

,Model,Model Name,Member,Calendar
0,access-cm2,ACCESS-CM2,r5i1p1f1,standard
1,canesm5,CanESM5,r1i1p2f1,365_day
2,cesm2,CESM2,r11i1p1f1,365_day
3,cnrm-esm2-1,CNRM-ESM2-1,r1i1p1f2,standard
4,ec-earth3,EC-Earth3,r1i1p1f1,standard
5,ec-earth3-veg,EC-Earth3-Veg,r1i1p1f1,standard
6,fgoals-g3,FGOALS-g3,r1i1p1f1,365_day
7,giss-e2-1-g,GISS-E2-1-G,r1i1p1f2,365_day
8,miroc6,MIROC6,r1i1p1f1,standard
9,mpi-esm1-2-hr,MPI-ESM1-2-HR,r3i1p1f1,standard


In [6]:
watershedpath = '/glade/u/home/tcorrie/'

# Load geopandas databases
wus_db = gpd.read_file(watershedpath+'watersheds_WUS.geojson', driver='GeoJSON')[['huc4', 'name', 'geometry']]

# Dimension names
dim0_name = "watershed_id"
dim1_name = "year"
dim2_name = "model"
dim3_name = "bc"
dim4_name = "ARDT"

def calc_mean_prec(watershed_idx, model_idx, bc_idx, ardt_idx):

    # Main loop
    ARDT = 'G18' if ardt_idx == 1 else 'TE' if ardt_idx == 2 else 'tARget' if ardt_idx == 3 else 0
    bcn = "_bc" if bc_idx else ""
    config = 'cmip6nonbc' if model_idx <= 15 and not bc_idx else 'cmip6bc' if model_idx <= 15 and bc_idx else 'cesm2le' if model_idx >= 16 else "--"

    modelrow = wusd3.loc[wusd3.index == model_idx-1]
    model = modelrow['Model'].item()
    name = modelrow['Model Name'].item()
    member = modelrow['Member'].item()
    calendar = modelrow['Calendar'].item()

    prec_path = "/glade/work/tcorrie/ARdata/pmean/"
    ARs_path = "/glade/work/tcorrie/ARdata/ARwatershed/"

    watershedrow = wus_db.loc[wus_db.index == watershed_idx-1]
    huc4id = int(watershedrow['huc4'].item())
    namestring = watershedrow['name'].item().replace(" ","_")
    pstring = f"ARDT: {ARDT}, BC: {bool(bc_idx)}, Model: {model}, Name: {name}, Member: {member}, Calendar: {calendar}, Watershed ID: {huc4id}, Namestring: {namestring}"
    print(f'{pstring:200}', end='\r')
    wshed_id = str(huc4id)
    
    prec_ds = xa.open_dataset(prec_path+f'prec_mean.{config}.{model}.{member}.nc', use_cftime=True)
    ARs_ds = xa.open_dataset(ARs_path+f'{ARDT}/ARwshed.{ARDT}.{model}.{member}.nc', use_cftime=True)
    
    prec_ds_wshed = prec_ds.sel(region=wshed_id)
    ARs_ds_wshed = ARs_ds.sel(watershed=wshed_id)[['ARwshed']]
    
    AR_cs_days = ARs_ds_wshed.where(((ARs_ds_wshed.time.dt.month >= 10) | (ARs_ds_wshed.time.dt.month <= 3)) & (ARs_ds_wshed.ARwshed == 1), drop=True).time.values
    prec_ds_wshed_AR_cs_days = prec_ds_wshed.sel(time=AR_cs_days)
    prec_ds_wshed_AR_cs_days['cold_season'] = xa.where(prec_ds_wshed_AR_cs_days.time.dt.month >= 10, prec_ds_wshed_AR_cs_days.time.dt.year, prec_ds_wshed_AR_cs_days.time.dt.year-1)
    groups = prec_ds_wshed_AR_cs_days['cold_season']

    def extreme_mean(ds, dim='time'):
        thold = ds.quantile(0.975, dim=dim)
        return ds.where(ds >= thold).mean(dim=dim)
    
    mean_annual_AR_precip = prec_ds_wshed_AR_cs_days.drop_vars('cold_season').groupby(groups).map(extreme_mean).prec_mean
        
    return mean_annual_AR_precip

2026-09-18 13:18:52,305 - distributed.utils_perf - WARNING - full garbage collections took 29% CPU time recently (threshold: 10%)


In [7]:
watersheds = np.arange(1, 25)
models = np.arange(1, 26)
bcs = [0, 1]
ardts = [1, 2, 3]
cs_years = np.arange(1980,2100)

ARprecip_means = xa.Dataset(coords=
                            {'watershed_idx': watersheds,
                             'model_idx': models,
                             'bc_idx': bcs,
                             'ardt_idx': ardts,
                             'cold_season': cs_years
                            },
                           data_vars={
                               'prec_mean': 
                               (['watershed_idx', 'model_idx', 'bc_idx', 'ardt_idx', 'cold_season'],
                                np.zeros(
                                    (len(watersheds),
                                     len(models),
                                     len(bcs),
                                     len(ardts),
                                     len(cs_years)
                                    )
                                )
                               )
                           }
                           )


for w in watersheds:
    for m in models:
        for b in bcs:
            for a in ardts:
                if m >= 16 and b==0:
                    ARprecip_means.loc[{'watershed_idx':w, 'model_idx':m, 'bc_idx':b, 'ardt_idx':a}] = np.nan
                else:
                    ARprecip_means.loc[{'watershed_idx':w, 'model_idx':m, 'bc_idx':b, 'ardt_idx':a}] = calc_mean_prec(w, m, b, a).reindex(cold_season=cs_years)



2026-09-18 13:19:03,448 - distributed.utils_perf - WARNING - full garbage collections took 34% CPU time recently (threshold: 10%)


2026-09-18 13:19:09,329 - distributed.utils_perf - WARNING - full garbage collections took 22% CPU time recently (threshold: 10%)


2026-09-18 13:19:19,481 - distributed.utils_perf - WARNING - full garbage collections took 22% CPU time recently (threshold: 10%)


2026-09-18 13:19:29,357 - distributed.utils_perf - WARNING - full garbage collections took 22% CPU time recently (threshold: 10%)


In [8]:
ARprecip_means.to_netcdf('mean_arrays/prec_extrememean_annually.nc')